# Exploração e resultados - Fase 3

Este notebook é uma interface de leitura. A lógica oficial fica em `src/` e `run_pipeline.py`. Execute primeiro o pipeline com a exportação real. Resultados do modo `--demo` não são evidência acadêmica.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.preprocessing.schema import load_dataset, prepare_target, select_features, validate_dataset

## 1. Contrato e panorama da entrada

In [ ]:
input_path = ROOT / 'data' / 'raw' / 'ml_alfabetizacao.csv'
frame = prepare_target(load_dataset(input_path))
validate_dataset(frame)
features = select_features(frame)
print(f'{len(frame):,} estudantes | {frame.id_municipio.nunique():,} municípios')
display((frame.nao_alfabetizado.value_counts(normalize=True).rename('proporcao').to_frame()))
display((frame[features.all].isna().mean().mul(100).sort_values(ascending=False).rename('% ausente').to_frame()))

## 2. Execute o pipeline reproduzível

A célula abaixo chama o mesmo entrypoint documentado no README.

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / 'run_pipeline.py'), '--input', str(input_path), '--leakage-demo'], check=True)

## 3. Compare modelos e avalie o teste

In [ ]:
validation = pd.read_csv(ROOT / 'data' / 'processed' / 'metricas_validacao.csv')
test = pd.read_csv(ROOT / 'data' / 'processed' / 'metricas_teste.csv')
display(validation.sort_values('pr_auc', ascending=False))
display(test)

## 4. Interpretação e municípios prioritários

In [ ]:
importance = pd.read_csv(ROOT / 'data' / 'processed' / 'importancia_variaveis.csv')
ranking = pd.read_csv(ROOT / 'data' / 'processed' / 'ranking_municipios.csv')
display(importance.head(15))
display(ranking.head(20))